In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
# n_epoch = 1000
n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.28905051946640015
epoch:  1, loss: 0.28312912583351135
epoch:  2, loss: 0.27704349160194397
epoch:  3, loss: 0.27147239446640015
epoch:  4, loss: 0.26610785722732544
epoch:  5, loss: 0.2609480619430542
epoch:  6, loss: 0.2558775246143341
epoch:  7, loss: 0.25114530324935913
epoch:  8, loss: 0.2463196963071823
epoch:  9, loss: 0.24168485403060913


In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2104.4191065175455
Test metrics:  R2 = -2098.2696278982935


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.45786094665527344
epoch:  1, loss: 0.25924766063690186
epoch:  2, loss: 0.05358182266354561
epoch:  3, loss: 0.045758895576000214
epoch:  4, loss: 0.02005508728325367
epoch:  5, loss: 0.013389681465923786
epoch:  6, loss: 0.010800380259752274
epoch:  7, loss: 0.009181275963783264
epoch:  8, loss: 0.008045317605137825
epoch:  9, loss: 0.00793475192040205


In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6919452003807034
Test metrics:  R2 = 0.5797226032985254


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonTR(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

TypeError: ExactBlockHessianCalculator.hvp() takes 2 positional arguments but 3 were given

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9931288957595825
Test metrics:  R2 = 0.9775075316429138
